In [0]:
from pyspark.sql import Row

# 8 sample "tables" as DataFrames, each with an assigned priority
sample_tables = [
    {"name": "customers",   "priority": 1, "df": spark.createDataFrame([Row(id=i) for i in range(100)])},
    {"name": "orders",      "priority": 1, "df": spark.createDataFrame([Row(id=i) for i in range(200)])},
    {"name": "products",    "priority": 2, "df": spark.createDataFrame([Row(id=i) for i in range(50)])},
    {"name": "payments",    "priority": 1, "df": spark.createDataFrame([Row(id=i) for i in range(150)])},
    {"name": "shipments",   "priority": 3, "df": spark.createDataFrame([Row(id=i) for i in range(80)])},
    {"name": "returns",     "priority": 2, "df": spark.createDataFrame([Row(id=i) for i in range(30)])},
    {"name": "inventory",   "priority": 1, "df": spark.createDataFrame([Row(id=i) for i in range(120)])},
    {"name": "suppliers",   "priority": 2, "df": spark.createDataFrame([Row(id=i) for i in range(40)])},
]
#iterating through the sample_tables
for t in sample_tables:
    print(f"{t['name']:12s} priority={t['priority']}  rows={t['df'].count()}")

In [0]:
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

#method describe on table 
def run_one(table):
    thread_name = threading.current_thread().name
    print(f"[START] {table['name']:12s} priority={table['priority']} on {thread_name} @ {time.strftime('%H:%M:%S')}")
    row_count = table["df"].count()   # stand-in for your real ingestion/processing logic
    time.sleep(1)                      # simulate extra work so overlap is visible
    print(f"[DONE ] {table['name']:12s} rows={row_count}")
    return {"name": table["name"], "priority": table["priority"], "rows": row_count, "status": "SUCCESS"}

In [0]:
#I am using ThreadPoolExecutor to run tasks concurrently
max_workers = 3  # fewer than 8 tasks so you can actually see queueing happen

tasks_sorted = sorted(sample_tables, key=lambda t: t["priority"])

print(f"Submission order (priority-sorted): {[t['name'] for t in tasks_sorted]}\n")

# Run tasks concurrently using ThreadPoolExecutor
results = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_task = {executor.submit(run_one, t): t for t in tasks_sorted}
    for future in as_completed(future_to_task):
        task = future_to_task[future]
        try:
            results.append(future.result())
        except Exception as exc:
            print(f"Task {task['name']} failed: {exc}")
            results.append({"name": task["name"], "priority": task["priority"], "status": "FAILED", "error": str(exc)})

print("\n--- Results ---")
for r in results:
    print(r)